# Fast-DetectGPT — Efficient Zero-Shot Detection via Conditional Probability Curvature

Bao et al., **ICLR 2024** ·
[proceedings.iclr.cc](https://proceedings.iclr.cc/paper_files/paper/2024/hash/6b8c6f846c3575e1d1ad496abea28826-Abstract-Conference.html) ·
arXiv: https://arxiv.org/abs/2310.05130 · code: https://github.com/baoguangsheng/fast-detect-gpt

> **Phạm vi**: chỉ inference + eval theo bộ metric của supervised. Chưa eval dưới attack.
> **Model 2/3 nhánh DetectGPT/Fast-DetectGPT** — kế thừa trực tiếp DetectGPT (model 1), thay bước
> perturb-and-rescore (T5, tốn K forward pass) bằng **kỳ vọng đóng dạng (closed-form)**, nhanh hơn
> ~340 lần theo paper. Không train — cấu hình chính dùng **GPT-J 6B làm sampling** và
> **GPT-Neo 2.7B làm scoring**, đúng cặp black-box mạnh trong Table 2 của paper.

## Cơ chế

DetectGPT cần sinh K bản nhiễu rồi tính log-prob từng bản (K forward pass tốn kém). Fast-DetectGPT nhận
ra: nếu model sampling đã biết rõ phân phối `p_sample(·|x_<t)` tại mỗi vị trí, có thể tính **kỳ vọng và
phương sai của `log p_score(Y)` khi `Y ~ p_sample`** bằng công thức đóng — không cần sample Monte-Carlo:

```
mean_t = Σ_v p_sample(v|x_<t) · log p_score(v|x_<t)
var_t  = Σ_v p_sample(v|x_<t) · (log p_score(v|x_<t))^2  -  mean_t^2
d(x)   = [Σ_t (log p_score(x_t|x_<t) - mean_t)] / sqrt(Σ_t var_t)   — Eq. (3),(6),(7)
```

Notebook này dùng `p_sample != p_score`: GPT-J cung cấp `p_sample`, GPT-Neo 2.7B cung cấp
`p_score`. Cả hai dùng tokenizer/vocabulary tương thích và mỗi model chỉ cần một forward pass cho
mỗi batch. Đây là **black-box surrogate setting** vì không truy cập source generator của dataset.

> **Ranh giới tái lập**: cặp model và phần score bám cấu hình black-box + nghiệm analytic của paper.
> Giới hạn 512 token, FP16 trên T4 và bộ metric chung là adapter thực nghiệm của project; không được
> hiểu là tái tạo nguyên xi các con số trên dataset gốc của paper.

## Deviation so với paper

| # | Paper | Ta dùng | Lý do |
|---|---|---|---|
| 1 | Black-box: GPT-J 6B sampling / GPT-Neo 2.7B scoring | **Khớp đúng cặp model** | Không deviation về backbone |
| 2 | Độ dài phụ thuộc thí nghiệm | `MAX_LEN=512` token | Đồng bộ ngân sách đầu vào với supervised và AL; tokenizer-specific |
| 3 | Paper mô tả model ≤6B chạy FP32 | Trọng số FP16 trên T4; softmax/kỳ vọng/phương sai FP32 | Deviation hạ tầng để mỗi model vừa một T4; công thức giữ nguyên |
| 4 | Sampling xấp xỉ hoặc nghiệm analytic | Nghiệm analytic chính xác, không giảm N | Khớp Appendix B; không có ngân sách Monte-Carlo phải đánh đổi |

GPT-J có output embedding được pad lớn hơn vocabulary tokenizer của GPT-Neo. Theo code gốc của tác
giả, khi hai kích thước logits khác nhau notebook cắt cả hai về `min(vocab_size)` trước khi tính
kỳ vọng/phương sai, đồng thời kiểm tra mọi label token ID vẫn nằm trong common vocabulary.

## Adapter so sánh với supervised và AL

- Cùng raw split `val` / `test_ood_model` / `test_fp`, cùng `melt_pairs` / `melt_fp` và `MAX_LEN=512`.
- `tpr@fpr*` giữ nguyên định nghĩa operating point trên ROC từng split để tương thích bảng hiện có.
- `TAU` chỉ calibrate trên human của `val`; `tpr@tau` và `fpr@tau` dùng đúng ngưỡng cố định đó.
- Prediction cuối giữ schema `paper_id,label,ai_degree,generator_family,score` như supervised/AL.
- Inference analytic là deterministic: không tạo mean±std giả theo seed; uncertainty nên bootstrap
  theo `paper_id` ở notebook tổng hợp.

⚡ **Dual T4 model-parallel**: một process điều khiển hai GPU — GPT-J nằm trên `cuda:0`, GPT-Neo
trên `cuda:1`; sampling logits được chuyển sang GPU scoring để tính Eq. (6),(7). Không dùng DDP hay
`torchrun`, vì nhân bản cả cặp model lên từng T4 sẽ OOM. FULL run có checkpoint theo chunk và tự
resume an toàn nếu session bị ngắt.


In [ ]:
!pip install -q -U "transformers>=4.45" accelerate sentencepiece

In [ ]:
%%writefile score_worker.py
import os, json, random, warnings, time, math, hashlib, gc
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score, average_precision_score

warnings.filterwarnings("ignore")

# ── Bản dataset dùng để train/eval ────────────────────────────────────
#   "deconf2"    = 02c_deconfound_v2.ipynb  <- ĐANG DÙNG
#                  canonicalize + truncate_pair_sentences + lọc notation_gap 85%.
#                  shortcut_audit.py đo đủ 22 đặc trưng: tầng process 0.5055 (cổng
#                  cứng, PASS) · tổng 0.8909. Train 7.025 cặp (85% của 8.265).
#   "deconf"     = 02b_deconfound.ipynb. ĐỪNG dùng cho nhánh này. Hai lý do:
#                  (a) con số "surface-GBM 0.6867, G6 PASS" mà 02b tự báo chỉ đo 6
#                      đặc trưng, thiếu avg_word_len (0.7795) lẫn type_token_ratio
#                      (0.7727). Đo đủ 22 thì là 0.9114 -> WARN, không PASS.
#                  (b) truncate_pair cắt thô đúng từ thứ min(len) nên chỉ 52,4% văn
#                      bản còn kết thúc đúng dấu câu (trước khi cắt: 99,0%). Detector
#                      ở file này chấm điểm bằng log-probability của từng token, nên
#                      token bị chặt giữa câu là perplexity GIẢ — và nó BẤT ĐỐI XỨNG,
#                      vì câu AI dài hơn (22,4 vs 20,4 từ) nên bên AI mất nhiều từ
#                      hơn một cách có hệ thống.
#   "normalized" = bản GỐC, CÒN shortcut: chỉ ĐẾM SỐ TỪ đã đoán đúng human/AI ở
#                  AUROC 0.7193 (surface-GBM 0.8080). G6 FAIL.
#
# PROVENANCE của số cũ (REPORT_2026-08-01.md §5) — đã truy được, ĐỪNG suy diễn lại:
# log còn lưu trong output notebook ghi sha256 val=c9d3957ae20b ood=2e9a47ca22e0
# fp=584cb43da181, khớp ĐÚNG dataset_{val,test_ood_model,test_fp}_normalized.parquet.
# Tức mọi số static/detectgpt trong báo cáo đó chạy trên "normalized" — chưa
# canonicalize, chưa cắt cặp — KHÔNG phải "deconf" như dòng tag cũ ghi (dòng đó nằm
# trên %%writefile nên chưa bao giờ tới được worker).
# Hệ quả: AUROC(val) < 0.5 của Fast-DetectGPT (0.4760) và Binoculars (0.4650) KHÔNG
# do tiền xử lý. Nguyên nhân phải tìm ở chỗ khác — mismatch thời đại backbone (gpt2-xl
# 2019 / gpt-j 2021 / falcon 2023 vs generator 2026), hoặc bug implementation. PHẢI có
# positive control trên một tập đã biết kết quả công bố trước khi tin bảng so sánh.
#
# PHẢI trùng tag với supervised_models/ và adversarial_models/ — cả 8 notebook đó đã
# chạy trên "deconf2". Lệch tag thì hai bảng số KHÔNG so được với nhau: deconf 8.265
# cặp vs deconf2 7.025 cặp, và nội dung text cũng khác (cắt theo từ vs cắt biên câu).
#
# PHẢI đặt TRONG body của %%writefile, KHÔNG phải ở đầu cell notebook. Đặt ở đầu cell
# thì %%writefile bị đẩy khỏi dòng 0 -> Jupyter hiểu thành line magic và báo "Line
# magic function `%%writefile` not found"; mà kể cả chạy được thì worker vẫn NameError
# vì nó là tiến trình riêng, không thấy biến của notebook.
#
# LƯU Ý: Kaggle dataset phải chứa dataset_{train,val,test_ood_model,test_fp}_deconf2.parquet
DATASET_TAG = "deconf2"
# Paper black-box setting: GPT-J sampling / GPT-Neo-2.7B scoring (Table 2).
IMPL_VERSION = "paper-blackbox-analytic-v4-2026-07-28"
SAMPLING_MODEL = "EleutherAI/gpt-j-6b"
SCORING_MODEL = "EleutherAI/gpt-neo-2.7B"
MAX_LEN = 512
BATCH_SIZE = 1       # an toàn VRAM cho GPT-J FP16 trên T4; thử 2 sau khi SMOKE ổn định
CHECKPOINT_ROWS = 100
RUN_NAME = "fast_detectgpt_gptj_neo27"
MODEL_DESC = SAMPLING_MODEL + "/" + SCORING_MODEL + " (sampling/scoring)"
EVAL_SETTING = "paper_black_box_surrogate"
RUN_SIGNATURE = "|".join([IMPL_VERSION, SAMPLING_MODEL, SCORING_MODEL,
                          str(MAX_LEN), "fp16-weights-fp32-statistics", EVAL_SETTING])

if not torch.cuda.is_available() or torch.cuda.device_count() < 2:
    raise RuntimeError("Can 2 GPU: GPT-J tren cuda:0 va GPT-Neo-2.7B tren cuda:1.")
SAMPLING_DEVICE = torch.device("cuda:0")
SCORING_DEVICE = torch.device("cuda:1")
SMOKE_TEST = os.environ.get("SMOKE", "0") == "1"
SMOKE_PER_CLASS = 16
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

def log(*a):
    print("[fastdetect]", *a, flush=True)

from transformers import AutoTokenizer, AutoModelForCausalLM

sampling_tok = AutoTokenizer.from_pretrained(SAMPLING_MODEL)
scoring_tok = AutoTokenizer.from_pretrained(SCORING_MODEL)
for _tok in {id(sampling_tok): sampling_tok, id(scoring_tok): scoring_tok}.values():
    if _tok.pad_token is None:
        _tok.pad_token = _tok.eos_token

# Load tuần tự với low_cpu_mem_usage để giảm đỉnh RAM host. Mỗi model nằm trọn trên một T4.
sampling_model = AutoModelForCausalLM.from_pretrained(
    SAMPLING_MODEL, torch_dtype=torch.float16, low_cpu_mem_usage=True).to(SAMPLING_DEVICE).eval()
sampling_model.config.use_cache = False
gc.collect(); torch.cuda.empty_cache()
scoring_model = AutoModelForCausalLM.from_pretrained(
    SCORING_MODEL, torch_dtype=torch.float16, low_cpu_mem_usage=True).to(SCORING_DEVICE).eval()
scoring_model.config.use_cache = False
gc.collect(); torch.cuda.empty_cache()
log("loaded:", SAMPLING_MODEL, "on", SAMPLING_DEVICE, "|", SCORING_MODEL, "on", SCORING_DEVICE)
log("impl:", IMPL_VERSION, "| setting:", EVAL_SETTING)
_vocab_crop_logged = False

@torch.inference_mode()
def score(texts, batch_size=BATCH_SIZE):
    """score cao = AI. Conditional probability curvature dạng đóng (Eq. Fast-DetectGPT)."""
    global _vocab_crop_logged
    out = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        score_enc_cpu = scoring_tok(batch, return_tensors="pt", truncation=True, max_length=MAX_LEN,
                                    padding=True, return_token_type_ids=False)
        sample_enc_cpu = sampling_tok(batch, return_tensors="pt", truncation=True,
                                      max_length=MAX_LEN, padding=True,
                                      return_token_type_ids=False)
        if (sample_enc_cpu["input_ids"].shape != score_enc_cpu["input_ids"].shape or
                not torch.equal(sample_enc_cpu["input_ids"], score_enc_cpu["input_ids"])):
            raise ValueError("GPT-J/GPT-Neo tokenizers khong tuong thich token IDs.")
        sample_enc = sample_enc_cpu.to(SAMPLING_DEVICE)
        score_enc = score_enc_cpu.to(SCORING_DEVICE)
        labels = score_enc["input_ids"][:, 1:]
        attn = score_enc["attention_mask"][:, 1:].float()

        # Hai forward pass được schedule trên hai GPU độc lập. use_cache=False giảm VRAM.
        sample_logits_gpu0 = sampling_model(**sample_enc, use_cache=False).logits[:, :-1, :]
        score_logits = scoring_model(**score_enc, use_cache=False).logits[:, :-1, :]
        # Code goc Fast-DetectGPT cat ve min vocab khi hai LM co output embedding padding khac nhau.
        # GPT-J thuong xuat 50,400 logits, GPT-Neo/GPT-2 tokenizer dung 50,257 token IDs.
        sample_vocab, score_vocab = sample_logits_gpu0.size(-1), score_logits.size(-1)
        common_vocab = min(sample_vocab, score_vocab)
        if int(labels.max().item()) >= common_vocab:
            raise ValueError(f"Label token ID vuot common_vocab={common_vocab}.")
        if sample_vocab != score_vocab and not _vocab_crop_logged:
            log(f"vocab logits mismatch {sample_vocab} vs {score_vocab}; "
                f"crop ca hai ve {common_vocab} nhu code goc Fast-DetectGPT.")
            _vocab_crop_logged = True
        sample_logits_gpu0 = sample_logits_gpu0[..., :common_vocab]
        score_logits = score_logits[..., :common_vocab]
        sample_logits = sample_logits_gpu0.to(SCORING_DEVICE, non_blocking=True)
        del sample_logits_gpu0

        logp_score = torch.log_softmax(score_logits.float(), dim=-1)
        p_sample = torch.softmax(sample_logits.float(), dim=-1)

        obs_logp = logp_score.gather(-1, labels.unsqueeze(-1)).squeeze(-1)
        mean_t = (p_sample * logp_score).sum(dim=-1)
        mean_t2 = (p_sample * logp_score.pow(2)).sum(dim=-1)
        var_t = (mean_t2 - mean_t.pow(2)).clamp(min=0.0)  # chỉ chặn sai số âm do floating point

        # Paper Eq. (3),(7): Var(sum_t Z_t) = sum_t Var(Z_t),
        # nen sigma cua TOAN CHUOI = sqrt(sum_t var_t), KHONG PHAI sum_t sqrt(var_t).
        num = ((obs_logp - mean_t) * attn).sum(dim=1)
        den = (var_t * attn).sum(dim=1).clamp(min=1e-12).sqrt()
        batch_scores = num / den
        if not torch.isfinite(batch_scores).all():
            raise FloatingPointError("Fast-DetectGPT sinh score NaN/Inf.")
        out.extend(batch_scores.float().cpu().tolist())
        del sample_logits, score_logits, logp_score, p_sample, obs_logp, mean_t, mean_t2, var_t
    return np.array(out)

# ── Data + checkpoint theo chunk; một process điều khiển cả hai GPU ──────────
KAGGLE_INPUT = "/kaggle/input/datasets/<kaggle-user>/halfdatadetector"

def find_data_dir():
    target = f"dataset_val_{DATASET_TAG}.parquet"
    if (Path(KAGGLE_INPUT) / target).exists():
        return Path(KAGGLE_INPUT)
    for base in [Path(KAGGLE_INPUT), Path("output"), Path("."), Path("/kaggle/input")]:
        if base.exists():
            hits = list(base.rglob(target))
            if hits:
                return hits[0].parent
    raise FileNotFoundError(f"Khong tim thay {target}. Kiem tra KAGGLE_INPUT = {KAGGLE_INPUT}")

DATA_DIR = find_data_dir()
OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
OUT_DIR = OUT_DIR / f"artifacts_{RUN_NAME}"
CHUNK_DIR = OUT_DIR / "chunks"
CHUNK_DIR.mkdir(parents=True, exist_ok=True)
log("DATA_DIR:", DATA_DIR, "| OUT_DIR:", OUT_DIR)

def sha256_file(path, block_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            block = f.read(block_size)
            if not block:
                break
            h.update(block)
    return h.hexdigest()

def melt_pairs(df):
    h = pd.DataFrame({"text": df["human_text"], "label": 0, "paper_id": df["paper_id"],
                       "ai_degree": 0, "generator_family": "human"})
    a = pd.DataFrame({"text": df["ai_text"], "label": 1, "paper_id": df["paper_id"],
                       "ai_degree": df.get("ai_degree", 3),
                       "generator_family": df.get("generator_family", "unknown")})
    out = pd.concat([h, a], ignore_index=True)
    return out[out["text"].notna() & (out["text"].astype(str).str.len() > 0)].reset_index(drop=True)

def melt_fp(df):
    has_ai = df["ai_text"].notna() if "ai_text" in df.columns else pd.Series(False, index=df.index)
    text = df["ai_text"].where(has_ai, df["human_text"]) if "ai_text" in df.columns else df["human_text"]
    out = pd.DataFrame({"text": text, "label": 0, "paper_id": df["paper_id"],
                         "ai_degree": df.get("ai_degree", 0).fillna(0).astype(int) if "ai_degree" in df.columns else 0,
                         "generator_family": "human"})
    return out[out["text"].notna() & (out["text"].astype(str).str.len() > 0)].reset_index(drop=True)

def load(name, fp_mode=False):
    p = DATA_DIR / f"dataset_{name}_{DATASET_TAG}.parquet"
    if not p.exists():
        log(f"thieu {p.name} - bo qua"); return None
    df = pd.read_parquet(p)
    return melt_fp(df) if fp_mode else melt_pairs(df)

def maybe_smoke(df):
    if df is None or not SMOKE_TEST:
        return df
    n = min(SMOKE_PER_CLASS, int(df["label"].value_counts().min()))
    return (df.groupby("label", group_keys=False)
              .sample(n=n, random_state=SEED)
              .sample(frac=1, random_state=SEED)
              .reset_index(drop=True))

def validate_chunk(part, path, split_name, data_sha, expected_n, chunk_idx, start, end):
    required = {"_row_id", "_run_signature", "_data_sha256", "_expected_n", "_chunk_idx",
                "paper_id", "label", "ai_degree", "generator_family", "score"}
    missing = required.difference(part.columns)
    ok = (not missing and set(part["_run_signature"].unique()) == {RUN_SIGNATURE}
          and set(part["_data_sha256"].unique()) == {data_sha}
          and set(part["_expected_n"].unique()) == {expected_n}
          and set(part["_chunk_idx"].unique()) == {chunk_idx}
          and part["_row_id"].tolist() == list(range(start, end))
          and np.isfinite(part["score"].to_numpy()).all())
    if not ok:
        raise RuntimeError(f"Checkpoint cu/hong: {path}. Doi ten OUT_DIR hoac xoa dung folder run nay.")

def score_split(split_name, fp_mode=False):
    data_path = DATA_DIR / f"dataset_{split_name}_{DATASET_TAG}.parquet"
    df = maybe_smoke(load(split_name, fp_mode=fp_mode))
    if df is None or len(df) == 0:
        return
    df["_row_id"] = np.arange(len(df), dtype=np.int64)
    if SMOKE_TEST:
        t0 = time.time()
        smoke_scores = score(df["text"].tolist())
        log(f"[SMOKE] {split_name}: {len(df)} dong, {time.time()-t0:.1f}s, "
            f"finite={np.isfinite(smoke_scores).all()}; KHONG luu checkpoint.")
        return

    data_sha = sha256_file(data_path)
    expected_n = len(df)
    n_chunks = math.ceil(expected_n / CHECKPOINT_ROWS)
    log(f"{split_name}: {expected_n} dong, {n_chunks} chunks, sha256={data_sha[:12]}...")
    split_t0 = time.time(); newly_scored = 0
    for chunk_idx in range(n_chunks):
        start = chunk_idx * CHECKPOINT_ROWS
        end = min(start + CHECKPOINT_ROWS, expected_n)
        out_path = CHUNK_DIR / f"{split_name}_chunk{chunk_idx:05d}.parquet"
        if out_path.exists():
            part = pd.read_parquet(out_path)
            validate_chunk(part, out_path, split_name, data_sha, expected_n, chunk_idx, start, end)
            log(f"resume {split_name} chunk {chunk_idx+1}/{n_chunks}: da co {len(part)} dong")
            continue

        part = df.iloc[start:end].copy()
        t0 = time.time()
        part["score"] = score(part["text"].tolist())
        part["_run_signature"] = RUN_SIGNATURE
        part["_data_sha256"] = data_sha
        part["_expected_n"] = expected_n
        part["_chunk_idx"] = chunk_idx
        chunk_cols = ["_row_id", "_run_signature", "_data_sha256", "_expected_n", "_chunk_idx",
                      "paper_id", "label", "ai_degree", "generator_family", "score"]
        tmp_path = out_path.with_suffix(".tmp.parquet")
        part[chunk_cols].to_parquet(tmp_path, index=False)
        os.replace(tmp_path, out_path)  # atomic: session ngat khong de lai checkpoint nua voi
        newly_scored += len(part)
        dt = time.time() - t0
        rate = len(part) / max(dt, 1e-9)
        remaining = expected_n - end
        log(f"saved {split_name} chunk {chunk_idx+1}/{n_chunks}: {len(part)} dong, "
            f"{rate:.2f} samples/s, ETA moi ~{remaining/max(rate,1e-9)/3600:.2f}h")
    log(f"XONG {split_name}: new={newly_scored}, wall={time.time()-split_t0:.1f}s")

for _split_name, _fp_mode in [("val", False), ("test_ood_model", False), ("test_fp", True)]:
    score_split(_split_name, fp_mode=_fp_mode)

for _dev in [0, 1]:
    log(f"peak VRAM cuda:{_dev} = {torch.cuda.max_memory_allocated(_dev)/1024**3:.2f} GiB")
log("XONG FULL worker." if not SMOKE_TEST else "XONG SMOKE worker.")

# ═══ CHẾ ĐỘ MIX — chấm tập trộn của 06_mixing để dựng đường dose-response ═════════
# Bật bằng: RUN_MIX=1 <lệnh chạy worker như thường>
#
# Vì sao nằm ở worker này chứ không ở 07_score_mix: 07 load checkpoint bằng
# AutoModelForSequenceClassification. Bốn detector zero-shot KHÔNG có checkpoint
# (falcon/gpt-j đóng băng), nên chúng phải được chấm bằng chính score() ở trên. Và PHẢI là
# score() đó, không viết lại — hai đường trong cùng một figure mà dùng hai hàm chấm khác
# nhau thì figure không nói được gì.
#
# Tập mix KHÔNG đi qua vòng split thường: text đã được 06 assemble sẵn nên không melt, và
# không có label nhị phân — chỉ có mix_ratio_actual liên tục. Vòng split thường tính
# AUROC/τ, vô nghĩa ở đây. Nên mix ghi riêng pred_mix*.parquet và KHÔNG vào metrics.json.
#
# Schema đầu ra = ĐÚNG 15 cột KEEP của 07_score_mix + score -> gộp chung được với
# scores_all.parquet mà không lệch cột.
MIX_NAME = "dataset_test_ood_model_mix.parquet"
MIX_KEEP = ["sample_id", "paper_id", "source", "generator_family", "mix_structure",
            "mix_seed", "mix_k_slots", "n_slots", "mix_ratio_actual", "n_seams",
            "word_count", "align_score", "was_mixed", "ai_degree", "in_spec_band"]
RUN_MIX = os.environ.get("RUN_MIX", "0") == "1"
# 0 = FULL (20.358 dòng). Đặt số n để lấy n dòng MỖI Ô (bin x mix_structure) = 20 ô.
MIX_PER_CELL = int(os.environ.get("MIX_PER_CELL", "0"))


def find_mix_file():
    """mix.parquet ở old_dataset/ khi chạy local, ở /kaggle/input khi chạy Kaggle."""
    for base in [Path(KAGGLE_INPUT), Path("/kaggle/input"), Path("."), Path(".."),
                 Path("old_dataset"), Path("output")]:
        try:
            if base.exists():
                hits = sorted(base.rglob(MIX_NAME))
                if hits:
                    return hits[0]
        except OSError:
            continue
    return None


def load_mix():
    p = find_mix_file()
    if p is None:
        log(f"khong thay {MIX_NAME} -> bo qua che do mix")
        return None, None
    df = pd.read_parquet(p)
    have = [c for c in MIX_KEEP if c in df.columns]
    missing = [c for c in MIX_KEEP if c not in df.columns]
    if missing:
        log(f"[!] mix thieu cot {missing} -> de NaN de concat voi 07 khong lech cot")
    out = df[["text"] + have].copy()
    for c in missing:
        out[c] = np.nan
    out = out[out["text"].notna() & (out["text"].astype(str).str.len() > 0)]
    if MIX_PER_CELL > 0:
        _b = (out["mix_ratio_actual"].fillna(0.0) * 10).astype(int).clip(0, 9)
        out = (out.assign(_bin=_b)
                  .groupby(["_bin", "mix_structure"], group_keys=False)
                  .apply(lambda g: g.sample(n=min(len(g), MIX_PER_CELL), random_state=SEED))
                  .drop(columns="_bin"))
        log(f"mix: subsample {MIX_PER_CELL} dong/o (bin x structure)")
    return out.reset_index(drop=True), p


def run_mix():
    import math as _math
    import time as _time

    _rank = int(globals().get("RANK", 0))
    _ws = int(globals().get("WORLD_SIZE", 1))
    df, src = load_mix()
    if df is None or len(df) == 0:
        return
    df["_row_id"] = np.arange(len(df), dtype=np.int64)
    shard = df.iloc[_rank::_ws].reset_index(drop=True) if _ws > 1 else df
    log(f"mix: {len(df)} dong tong -> {len(shard)} dong cua rank {_rank}/{_ws}  <- {src}")
    if SMOKE_TEST:
        _s = score(shard["text"].tolist()[:8])
        log(f"[SMOKE] mix: 8 dong, finite={np.isfinite(_s).all()}, "
            f"min/max={float(np.min(_s)):.4f}/{float(np.max(_s)):.4f}; KHONG luu.")
        return

    ck = OUT_DIR / f"chunks_mix_rank{_rank}"
    ck.mkdir(parents=True, exist_ok=True)
    step = int(globals().get("CHECKPOINT_ROWS", 0) or 200)
    n_ch = _math.ceil(len(shard) / step)
    log(f"mix: {n_ch} chunk x {step} dong, checkpoint tai {ck}")
    t_all = _time.time()
    for k in range(n_ch):
        a, b = k * step, min((k + 1) * step, len(shard))
        p = ck / f"mix_chunk{k:05d}.parquet"
        if p.exists():
            log(f"resume mix chunk {k + 1}/{n_ch}: da co")
            continue
        part = shard.iloc[a:b].copy()
        t0 = _time.time()
        part["score"] = score(part["text"].tolist())
        tmp = p.with_suffix(".tmp.parquet")
        part.drop(columns=["text"]).to_parquet(tmp, index=False)
        os.replace(tmp, p)          # atomic -> het gio session khong de lai chunk viet do dang
        rate = (b - a) / max(_time.time() - t0, 1e-9)
        log(f"mix chunk {k + 1}/{n_ch}: {rate:.2f} mau/s, "
            f"ETA con lai ~{(len(shard) - b) / max(rate, 1e-9) / 3600:.2f}h")

    parts = sorted(ck.glob("mix_chunk*.parquet"))
    if parts:
        allp = pd.concat([pd.read_parquet(x) for x in parts], ignore_index=True)
        dst = OUT_DIR / (f"pred_mix_rank{_rank}.parquet" if _ws > 1 else "pred_mix.parquet")
        allp.to_parquet(dst, index=False)
        log(f"da luu {dst.name}: {len(allp)} dong, wall={_time.time() - t_all:.0f}s")
        log(f"     cot: {list(allp.columns)}")


if RUN_MIX:
    run_mix()

## Chạy trên dual T4 (Kaggle)

`score_worker.py` chạy **một process model-parallel**: GPT-J trên GPU 0, GPT-Neo trên GPU 1.
Không dùng `torchrun`; mỗi GPU giữ đúng một model để tránh OOM. Kết quả FULL được checkpoint mỗi
`CHECKPOINT_ROWS=100` dòng và tự resume khi chạy lại cùng cell.

1. **SMOKE**: tải hai model, chấm mẫu nhỏ cả ba split, không lưu checkpoint.
2. **FULL**: chấm toàn bộ `val`/`test_ood_model`/`test_fp`; chạy lại sẽ bỏ qua chunk hợp lệ đã có.
3. Chạy cell **gộp chunk** rồi cell **metrics**.

⚠️ Notebook yêu cầu đúng 2 GPU. Không chuyển sang `torchrun --nproc_per_node=2`: cách đó nhân bản
GPT-J + GPT-Neo trên mỗi T4 và sẽ OOM.


In [ ]:
# ── Checkpoint của lần chạy trước: PHẢI xoá khi đổi DATASET_TAG ───────────────
# Chunk/partial cũ mang _data_sha256 của file parquet CŨ. Notebook nào có
# validate_chunk() thì raise; notebook nào không có thì lẳng lặng gộp score của
# dataset cũ vào metric của dataset mới — đúng loại bug im lặng tệ nhất.
#
# Cell này KHÔNG tự xoá. Nó liệt kê ra để bạn nhìn trước, rồi bạn bật cờ.
import shutil
from pathlib import Path

PURGE_STALE = False   # đặt True MỘT LẦN sau khi đổi tag, chạy cell, rồi trả lại False

_base = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
_ckpt, _derived = [], []
for _d in sorted(_base.glob("artifacts_*")):
    _ckpt += [p for p in (_d / "chunks",) if p.is_dir()]
    _ckpt += sorted(_d.glob("partial_*.parquet")) + sorted(_d.glob("chunk_*.parquet"))
    _derived += sorted(_d.glob("pred_*.parquet")) + sorted(_d.glob("metrics*.json"))

if not _ckpt and not _derived:
    print("sạch — không có kết quả của lần chạy trước.")
else:
    print(f"CHECKPOINT (score thô, phải xoá): {len(_ckpt)}")
    for _p in _ckpt:
        print("   ", _p)
    print(f"DẪN XUẤT (metric/pred, bị ghi đè khi chạy lại đủ 3 split): {len(_derived)}")
    for _p in _derived:
        print("   ", _p)
    if PURGE_STALE:
        for _p in _ckpt + _derived:
            shutil.rmtree(_p) if _p.is_dir() else _p.unlink()
        print(f"\nĐÃ XOÁ {len(_ckpt) + len(_derived)} mục.")
    else:
        print("\nPURGE_STALE = False -> chưa xoá gì. Bật True rồi chạy lại cell này.")

In [ ]:
# ── SMOKE: kiểm 2 GPU bắt tay + đọc được parquet, vài phút ──────────────────────
# RUN_MIX=1 ở đây để SMOKE kiểm LUÔN việc tìm thấy dataset_test_ood_model_mix.parquet
# và chấm được 8 dòng của nó. Không có nó thì bạn chỉ phát hiện "quên upload mix" SAU
# khi benchmark đã chạy hàng giờ — worker bỏ qua mix im lặng, không báo lỗi.
# Phải thấy 3 dòng sha256/số dòng của deconf2 VÀ một dòng "[SMOKE] mix: 8 dong".
# Cell này đang để COMMENT có chủ ý: trong lần Save Version (Batch) thì SMOKE
# ăn mất ~10 phút của ngân sách 12h. Bỏ dấu # khi chạy ở session Interactive.
# !RUN_MIX=1 SMOKE=1 OMP_NUM_THREADS=2 python3 score_worker.py

In [ ]:
# ── CHẠY FULL: 3 split benchmark + tập MIX trong MỘT lần gọi ────────────────────
# RUN_MIX=1 nằm ở đây thay vì một cell riêng vì worker load model ở module level. Tách
# hai lệnh thì Fast-DetectGPT load model HAI lần; gộp lại thì load một lần rồi chấm
# splits -> mix liền. Block mix nằm cuối lastde/score_worker.py nên thứ tự tự đúng.
#
# Ước tính: ~1h + ~2,3h = ~3,3h. Dưới hạn 12h/session của Kaggle.
# Checkpoint chunk + atomic replace ở CẢ HAI phần -> session chết thì chạy lại cell này
# là tiếp tục đúng chỗ, không chấm lại gì.
# Muốn rút tập mix: thêm MIX_PER_CELL=150 (150 dòng mỗi ô bin x structure = 3.000 dòng).
!RUN_MIX=1 OMP_NUM_THREADS=2 python3 score_worker.py

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score

IMPL_VERSION = "paper-blackbox-analytic-v4-2026-07-28"
RUN_NAME = "fast_detectgpt_gptj_neo27"
MODEL_DESC = "EleutherAI/gpt-j-6b/EleutherAI/gpt-neo-2.7B (sampling/scoring)"
EVAL_SETTING = "paper_black_box_surrogate"
RUN_SIGNATURE = "|".join([IMPL_VERSION, "EleutherAI/gpt-j-6b", "EleutherAI/gpt-neo-2.7B",
                          "512", "fp16-weights-fp32-statistics", EVAL_SETTING])
TARGET_FPR = 0.01
CHECKPOINT_ROWS = 100

def _merge_chunks(split_name):
    paths = sorted(CHUNK_DIR.glob(f"{split_name}_chunk*.parquet"))
    if not paths:
        raise FileNotFoundError(f"Chua co checkpoint FULL cho split {split_name}.")
    parts = []
    for p in paths:
        part = pd.read_parquet(p)
        required = {"_row_id", "_run_signature", "_data_sha256", "_expected_n", "_chunk_idx",
                    "paper_id", "label", "ai_degree", "generator_family", "score"}
        missing = required.difference(part.columns)
        if missing:
            raise RuntimeError(f"{p.name} thieu cot integrity: {sorted(missing)}")
        if set(part["_run_signature"].unique()) != {RUN_SIGNATURE}:
            raise RuntimeError(f"{p.name} thuoc config/implementation cu; hay chay lai FULL.")
        if not np.isfinite(part["score"].to_numpy()).all():
            raise RuntimeError(f"{p.name} co score NaN/Inf.")
        parts.append(part)
    expected_values = {int(x) for part in parts for x in part["_expected_n"].unique()}
    if len(expected_values) != 1:
        raise RuntimeError(f"Cac chunk {split_name} khong cung expected_n: {expected_values}")
    expected_n = expected_values.pop()
    data_hashes = {str(x) for part in parts for x in part["_data_sha256"].unique()}
    if len(data_hashes) != 1:
        raise RuntimeError(f"Cac chunk {split_name} den tu nhieu ban dataset: {data_hashes}")
    merged = pd.concat(parts, ignore_index=True)
    if len(merged) != expected_n:
        raise RuntimeError(f"{split_name}: gop duoc {len(merged)} != expected {expected_n} dong.")
    if merged["_row_id"].duplicated().any() or set(merged["_row_id"]) != set(range(expected_n)):
        raise RuntimeError(f"{split_name}: _row_id bi thieu hoac trung giua cac chunk.")
    return merged.sort_values("_row_id").reset_index(drop=True)

OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
OUT_DIR = OUT_DIR / f"artifacts_{RUN_NAME}"
CHUNK_DIR = OUT_DIR / "chunks"

val_df = _merge_chunks("val")
ood_df = _merge_chunks("test_ood_model")
fp_df = _merge_chunks("test_fp")
for nm, d in [("val", val_df), ("test_ood_model", ood_df), ("test_fp", fp_df)]:
    if d is not None:
        print(f"{nm:<16} {len(d):>7,} mau (gop chunk)  label={d['label'].value_counts().to_dict()}")


In [ ]:
# ── Metrics: CÙNG schema/công thức với supervised + AL ───────────────────────
# score() đã được tính sẵn trong score_worker.py (dual-GPU) và gộp lại ở cell trên.
# tpr@fpr* = operating point trên ROC của chính split (metric ranking, giữ để so sánh chung).
# tpr@tau/fpr@tau = dùng TAU cố định calibrate duy nhất trên val (metric deployment).
def tpr_at_fpr(y, s, fpr):
    y, s = np.asarray(y), np.asarray(s); neg = s[y == 0]
    if len(neg) == 0 or (y == 1).sum() == 0:
        return np.nan, np.nan
    thr = np.quantile(neg, 1 - fpr, method="higher")
    return float((s[y == 1] >= thr).mean()), float(thr)

def auroc(y, s):
    y = np.asarray(y)
    return float(roc_auc_score(y, s)) if len(np.unique(y)) == 2 else np.nan

def report_split(name, d):
    if d is None:
        return None
    y, s = d["label"].values, d["score"].values
    row = {"split": name, "n": len(d), "n_ai": int((y == 1).sum()),
           "auroc": auroc(y, s),
           "auprc": float(average_precision_score(y, s)) if len(np.unique(y)) == 2 else np.nan}
    for f in [0.001, 0.01, 0.05]:
        row[f"tpr@fpr{f}"] = tpr_at_fpr(y, s, f)[0]
    row["tpr@tau"] = float((s[y == 1] >= TAU).mean()) if (y == 1).any() else np.nan
    row["fpr@tau"] = float((s[y == 0] >= TAU).mean()) if (y == 0).any() else np.nan
    return row

# Paper định nghĩa curvature cao hơn => giống machine-generated hơn.
# KHÔNG được dùng nhãn val để tự lật dấu: AUROC < 0.5 là kết quả yếu hoặc lỗi cần điều tra.
_std = float(np.std(val_df["score"])) if val_df is not None else 0.0
_auc = auroc(val_df["label"], val_df["score"]) if val_df is not None else float("nan")
print(f"std(score val) = {_std:.4f} | AUROC(val) = {_auc:.4f} | score_direction=paper_fixed")
if not np.isnan(_auc) and _auc < 0.5:
    print("[!] AUROC(val) < 0.5; GIU NGUYEN chieu score theo paper. Kiem tra backbone mismatch/scoring.")
if _std < 1e-6 or (not np.isnan(_auc) and _auc < 0.55):
    print("[!] CANH BAO: score gan nhu hang so / AUROC~0.5 - kiem tra lai model/scoring truoc khi tin bang so sanh.")

SMOKE_TEST = False   # cell merge chỉ nên chạy sau FULL; đổi True nếu bạn đang gộp kết quả SMOKE để debug
_, TAU = tpr_at_fpr(val_df["label"], val_df["score"], TARGET_FPR) if val_df is not None else (np.nan, np.nan)
print(f"tau (FPR={TARGET_FPR:.0%} tren val) = {TAU:.4f}")

rows = [r for r in [report_split("val", val_df), report_split("test_ood_model", ood_df),
                     report_split("test_fp", fp_df)] if r]
metrics_df = pd.DataFrame(rows).set_index("split")
pd.set_option("display.width", 200, "display.max_columns", 20)
print(metrics_df.round(4).to_string())

if not SMOKE_TEST:
    metrics_out = {"run": RUN_NAME, "model": MODEL_DESC, "method": "fast_detectgpt",
                   "implementation": IMPL_VERSION, "paper_analytic_formula": True,
                   "eval_setting": EVAL_SETTING, "score_direction": "paper_fixed_high_is_ai",
                   "tau": TAU, "tau_source": "val_human", "target_fpr": TARGET_FPR,
                   "tpr_at_fpr_definition": "split_local_roc_operating_point",
                   "fixed_tau_metrics": ["tpr@tau", "fpr@tau"],
                   "seed": 42, "deterministic_inference": True, "flipped_sign": False,
                   "training_free": True, "precision": "fp16_weights_fp32_statistics",
                   "gpu_layout": {"cuda:0": "gpt-j-6b_sampling",
                                  "cuda:1": "gpt-neo-2.7B_scoring"},
                   "processes": 1, "gpu_count": 2, "checkpoint_rows": CHECKPOINT_ROWS,
                   "metrics": metrics_df.reset_index().to_dict("records")}
    (OUT_DIR / "metrics.json").write_text(json.dumps(metrics_out, indent=2, default=float))
    for nm, d in [("val", val_df), ("test_ood_model", ood_df), ("test_fp", fp_df)]:
        if d is not None:
            d[["paper_id", "label", "ai_degree", "generator_family", "score"]].to_parquet(
                OUT_DIR / f"pred_{nm}.parquet", index=False)
    print("da luu artifacts (gop chunks) ->", OUT_DIR)


In [ ]:
# 📊 Tóm tắt nhanh (đọc lại metrics.json vừa lưu)
import json as _json
from pathlib import Path as _Path
_p = OUT_DIR / "metrics.json"
if _p.exists():
    m = _json.loads(_p.read_text())
    print(f"RUN = {m['run']} | model = {m['model']} | setting = {m['eval_setting']}")
    print(f"impl = {m['implementation']} | tau(val,FPR=1%) = {m['tau']:.4f} | direction = {m['score_direction']}")
    for r in m["metrics"]:
        print(f"{r['split']:16} AUROC={r['auroc']:.4f}  ROC-TPR@FPR1%={r['tpr@fpr0.01']:.4f}  "
              f"TPR@val-tau={r['tpr@tau']:.4f}  FPR@val-tau={r['fpr@tau']:.4f}")
else:
    print("Chưa có metrics.json — chạy cell FULL RUN trước (bỏ SMOKE=1).")


In [ ]:
# ── CHẨN ĐOÁN "CÓ SO SÁNH ĐƯỢC KHÔNG" — chạy sau cell metric ─────────────────
# Bảng metric ở trên không trả lời được ba câu mà bảng benchmark cuối cùng phải trả lời:
#
#   1. CHIỀU có nhất quán giữa các split? val và test_ood_model cùng phía 0.5 -> không có
#      bug đảo dấu. Nằm hai phía -> không có tín hiệu, và đừng đọc dấu của từng split.
#   2. AUROC có khác ĐOÁN MÒ không? SD của AUROC khi detector vô hiệu là
#      sqrt((n1+n2+1)/(12*n1*n2)) — với val 438+438 thì SD = 0.0195, nên 0.48 và 0.52
#      là CÙNG một kết luận: không phân biệt được.
#   3. TPR@FPR=1% so với mốc nào? Mốc đoán mò là ĐÚNG 0.01 (ROC của classifier vô hiệu là
#      đường chéo), KHÔNG phải 0. Nên 0.0337 là 3.4x đoán mò, không phải "gần như trượt".
#
# In cả mean score theo nhãn: nếu mean(AI) > mean(human) ở CẢ BA split thì hướng đăng ký
# trước là đúng và số thấp là bản chất, không phải lỗi implementation.
import glob
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

_CHANCE = {"tpr@fpr0.001": 0.001, "tpr@fpr0.01": 0.01, "tpr@fpr0.05": 0.05}


def _sd_null(n1, n2):
    return math.sqrt((n1 + n2 + 1) / (12 * n1 * n2))


def _p2(z):
    return math.erfc(abs(z) / math.sqrt(2))


_preds = sorted(p for p in glob.glob("**/pred_*.parquet", recursive=True)
                if "pred_matched" not in p)
if not _preds:
    print("khong tim thay pred_*.parquet -> chay cell metric truoc")
else:
    print("=" * 92)
    print("CHIEU SCORE THEO NHAN (mean; huong dang ky: cao = AI)")
    print("=" * 92)
    print(f"  {'split':18}{'n':>7}{'mean human':>13}{'mean AI':>12}{'gap':>11}  dung chieu?")
    _signs = {}
    for _p in _preds:
        _d = pd.read_parquet(_p).dropna(subset=["score"])
        _sp = Path(_p).stem.replace("pred_", "")
        _h = _d.loc[_d["label"] == 0, "score"]
        _a = _d.loc[_d["label"] == 1, "score"]
        if len(_a) == 0:
            print(f"  {_sp:18}{len(_d):>7}{_h.mean():13.4f}{'—':>12}{'—':>11}  (1 lop, bo qua)")
            continue
        _gap = _a.mean() - _h.mean()
        _signs[_sp] = _gap
        print(f"  {_sp:18}{len(_d):>7}{_h.mean():13.4f}{_a.mean():12.4f}{_gap:+11.4f}"
              f"  {'CO' if _gap > 0 else 'KHONG'}")
    if len(_signs) >= 2:
        _ok = all(v > 0 for v in _signs.values()) or all(v < 0 for v in _signs.values())
        print(f"\n  -> chieu {'NHAT QUAN' if _ok else 'KHONG nhat quan'} qua "
              f"{len(_signs)} split"
              + ("" if _ok else "  <-- dau hieu KHONG CO TIN HIEU, dung doc dau tung split"))
        if _ok and all(v > 0 for v in _signs.values()):
            print("     dung huong dang ky truoc => implementation dung chieu; so thap la ban chat")

    print()
    print("=" * 92)
    print("CO HON DOAN MO KHONG")
    print("=" * 92)
    print(f"  {'split':18}{'AUROC':>9}{'SD_null':>10}{'z':>8}{'p':>9}  ket luan")
    for _p in _preds:
        _d = pd.read_parquet(_p).dropna(subset=["score"])
        _sp = Path(_p).stem.replace("pred_", "")
        _n1 = int((_d["label"] == 1).sum())
        _n0 = int((_d["label"] == 0).sum())
        if _n1 == 0 or _n0 == 0:
            continue
        _a = float(roc_auc_score(_d["label"], _d["score"]))
        _sd = _sd_null(_n0, _n1)
        _z = (_a - 0.5) / _sd
        _v = "hon ngau nhien" if (_p2(_z) < 0.05 and _z > 0) else (
            "DUOI ngau nhien" if _p2(_z) < 0.05 else "KHONG phan biet duoc voi doan mo")
        print(f"  {_sp:18}{_a:9.4f}{_sd:10.4f}{_z:+8.2f}{_p2(_z):9.4f}  {_v}")

    print()
    print("=" * 92)
    print("TPR SO MOC DOAN MO (moc = dung bang FPR, khong phai 0)")
    print("=" * 92)
    _art = Path(_preds[0]).parent
    _mp = _art / "metrics.json"
    _m = json.loads(_mp.read_text()) if _mp.exists() else None
    if _m is None:
        print("  khong doc duoc metrics.json")
    else:
        print(f"  {'split':18}{'chi so':16}{'gia tri':>10}{'doan mo':>10}{'lan':>7}{'z':>8}")
        for _r in _m["metrics"]:
            _npos = _r.get("n_ai") or 0
            _nneg = (_r.get("n") or 0) - _npos
            if not _npos or not _nneg:
                continue
            for _k, _c in _CHANCE.items():
                _v = _r.get(_k)
                if _v is None or (isinstance(_v, float) and math.isnan(_v)):
                    continue
                _z = (_v - _c) / math.sqrt(_c * (1 - _c) / _npos)
                # nguong FPR duoc dat boi phan vi tren NEGATIVE -> so mau quyet dinh = _c*_nneg
                _ndec = _c * _nneg
                _flag = f"  <-- nguong do ~{_ndec:.1f} mau quyet dinh, DUNG BAO" if _ndec < 3 else ""
                print(f"  {_r['split']:18}{_k:16}{_v:10.4f}{_c:10.4f}{_v / _c:6.1f}x{_z:+8.1f}{_flag}")